# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [15]:
import os
import subprocess

if not os.path.exists("flyrank-ml-starter"):
    !git clone https://github.com/Nikita-Sudarshan/flyrank-ml-starter.git

print("Current directory:", os.getcwd())
print("Repository exists:", os.path.exists("flyrank-ml-starter"))

Current directory: /content
Repository exists: True


In [16]:
import os
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

print("Imports loaded successfully.")

Imports loaded successfully.


In [17]:
# Find the FlyRank repository and dataset

repo_path = "/content/flyrank-ml-starter"
data_path = os.path.join(
    repo_path,
    "data",
    "raw",
    "content_refresh_anonymized.csv"
)

if not os.path.exists(data_path):
    raise FileNotFoundError(
        "content_refresh_anonymized.csv was not found at:\n"
        + data_path
    )

df = pd.read_csv(data_path)

print("Loaded from:", data_path)
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("\nColumns:")
print(list(df.columns))

Loaded from: /content/flyrank-ml-starter/data/raw/content_refresh_anonymized.csv
Rows: 30000
Columns: 44

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [18]:
print("Dataset shape:", df.shape)

print("\nTarget values:")
print(df["trend_direction"].value_counts())

print("\nTrend-related columns:")
print([
    col for col in df.columns
    if "trend" in col.lower()
])

Dataset shape: (30000, 44)

Target values:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Trend-related columns:
['trend_direction', 'trend_pct']


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [19]:
# ## 1. Build the feature vector

# The prediction target is whether the observed trend direction is `down`.

# The binary target is defined as:

# - `1` = observed trend direction is `down`
# - `0` = all other observed trend directions

# The feature vector uses the available content, search, engagement, age, and performance fields.

# `trend_direction` is excluded because it is the target. `trend_pct` is excluded because it is directly related to the target and creates a leakage risk. `client_id` is excluded from the model features because it identifies the client and is used only for grouped validation.

# Numeric missing values are filled with the median. Categorical missing values are filled with the most frequent category and then one-hot encoded.

In [20]:
# Target
y = df["trend_direction"].eq("down").astype(int)

# Explicit exclusions
excluded_columns = [
    "trend_direction",
    "trend_pct",
    "client_id"
]

# Build raw feature matrix
X = df.drop(columns=excluded_columns).copy()

print("Target: trend_direction → 1 = down, 0 = other")
print("\nTarget distribution:")
print(y.value_counts())

print("\nExcluded columns:")
print(excluded_columns)

print("\nFeature matrix shape:", X.shape)
print("Total raw model features:", X.shape[1])

print("\nFeature columns:")
print(list(X.columns))

Target: trend_direction → 1 = down, 0 = other

Target distribution:
trend_direction
1    16262
0    13738
Name: count, dtype: int64

Excluded columns:
['trend_direction', 'trend_pct', 'client_id']

Feature matrix shape: (30000, 41)
Total raw model features: 41

Feature columns:
['content_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_ti

In [21]:
numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude=["number"]
).columns.tolist()

print("Numeric features:", len(numeric_features))
print(numeric_features)

print("\nCategorical features:", len(categorical_features))
print(categorical_features)

Numeric features: 29
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']

Categorical features: 12
['content_id', 'competition_level', 'content_type', 'main_intent', 'provider_used', 'model_used', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier']


In [22]:
# Numeric preprocessing:
# missing numeric values → median

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

# Categorical preprocessing:
# missing categorical values → most frequent
# categories → one-hot encoded
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ]
)

print("Preprocessing pipeline created.")

Preprocessing pipeline created.


In [23]:
X_vector = preprocessor.fit_transform(X)

print("Raw feature count:", X.shape[1])
print("Processed feature-vector shape:", X_vector.shape)
print("Feature vector created successfully.")

Raw feature count: 41
Processed feature-vector shape: (30000, 30071)
Feature vector created successfully.


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [24]:
# ## 2. Feature notes (meaning, missing, categorical, available-when?)

# The features represent information that can reasonably be observed before a content-review decision.

# Search features describe demand and competition. Content features describe page type and content size. Historical performance features describe impressions, clicks, sessions, and engagement over defined historical windows. Age and freshness features describe how old or recently updated the content is.

# Numeric missing values are handled using median imputation. Categorical missing values are handled using the most frequent observed category and then one-hot encoded.

# The model does not use `client_id` as a predictive feature. It is retained separately for grouped validation.

# The target and `trend_pct` are not available as prediction features because they define or directly relate to the observed outcome being predicted.

In [25]:
feature_notes = pd.DataFrame({
    "feature": X.columns,
    "data_type": [
        str(X[col].dtype) for col in X.columns
    ],
    "missing_count": [
        int(X[col].isna().sum()) for col in X.columns
    ],
    "missing_handling": [
        "median" if col in numeric_features else "most_frequent"
        for col in X.columns
    ],
    "categorical": [
        col in categorical_features
        for col in X.columns
    ]
})

display(feature_notes)

,feature,data_type,missing_count,missing_handling,categorical
0,content_id,object,0,most_frequent,True
1,search_volume,float64,2468,median,False
2,competition,float64,2468,median,False
3,competition_level,object,2610,most_frequent,True
4,cpc,float64,2468,median,False
5,content_type,object,0,most_frequent,True
6,main_intent,object,2374,most_frequent,True
7,word_count,float64,7699,median,False
8,char_count,float64,7699,median,False
9,provider_used,object,21438,most_frequent,True


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [26]:
# ## 3. The leakage hunt

# The main leakage risk is allowing information derived from the observed outcome into the feature vector.

# The target field `trend_direction` must not appear in the features.

# `trend_pct` is explicitly excluded because it is directly related to the trend outcome.

# `client_id` is excluded from model features because it identifies the client and could allow the model to learn client-specific patterns rather than general content signals.

# Historical windows such as the previous 30-day period are retained because they represent prior observed performance rather than the target itself.

# The feature set is checked programmatically for target and excluded leakage-risk columns.

In [27]:
# Leakage / exclusion checks

target_column = "trend_direction"

leakage_risk_columns = [
    "trend_direction",
    "trend_pct",
    "client_id"
]

print("Checking feature vector for leakage-risk columns:\n")

for column in leakage_risk_columns:
    if column in X.columns:
        print("FAIL —", column, "is present in model features.")
    else:
        print("PASS —", column, "is not in model features.")

assert target_column not in X.columns
assert "trend_pct" not in X.columns
assert "client_id" not in X.columns

print("\nPASS — target and explicitly excluded leakage-risk columns")
print("are not present in the feature vector.")

Checking feature vector for leakage-risk columns:

PASS — trend_direction is not in model features.
PASS — trend_pct is not in model features.
PASS — client_id is not in model features.

PASS — target and explicitly excluded leakage-risk columns
are not present in the feature vector.


In [28]:
# Search feature names for obvious future/outcome-style fields

future_keywords = [
    "future",
    "next_",
    "post_",
    "after_",
    "outcome",
    "trend"
]

possible_future_fields = [
    col for col in X.columns
    if any(keyword in col.lower() for keyword in future_keywords)
]

print("Potential future/outcome-style feature names found:")
print(possible_future_fields)

if possible_future_fields:
    print(
        "\nThese fields require manual review; "
        "they are not automatically treated as leakage."
    )
else:
    print("\nNo obvious future/outcome-style feature names found.")

Potential future/outcome-style feature names found:
[]

No obvious future/outcome-style feature names found.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [29]:
# ## 4. What I excluded and why

# | Field | Reason for exclusion |
# |---|---|
# | `trend_direction` | This is the prediction target and cannot be used as a feature. |
# | `trend_pct` | Directly related to the observed trend outcome and therefore creates a leakage risk. |
# | `client_id` | Client identifier; excluded from predictive features and retained only for grouped validation. |

# The exclusions are deliberate and are verified programmatically above.

In [30]:
print("Final feature audit\n")

print("Target:")
print(" -", target_column)

print("\nExplicitly excluded:")
for column in leakage_risk_columns:
    print(" -", column)

print("\nFinal model feature count:", len(X.columns))

print("\nChecking excluded columns:")

all_passed = True

for column in leakage_risk_columns:
    passed = column not in X.columns
    print(
        ("PASS" if passed else "FAIL"),
        "—",
        column
    )

    if not passed:
        all_passed = False

print(
    "\n"
    + (
        "ALL LEAKAGE CHECKS PASSED"
        if all_passed
        else "LEAKAGE CHECK FAILED"
    )
)

Final feature audit

Target:
 - trend_direction

Explicitly excluded:
 - trend_direction
 - trend_pct
 - client_id

Final model feature count: 41

Checking excluded columns:
PASS — trend_direction
PASS — trend_pct
PASS — client_id

ALL LEAKAGE CHECKS PASSED


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [31]:
print("ML-05 FINAL SELF-CHECK\n")

checks = {
    "Dataset has 30,000 rows": len(df) == 30000,
    "Dataset has 44 columns": len(df.columns) == 44,
    "Target exists": "trend_direction" in df.columns,
    "Raw feature count is 41": len(X.columns) == 41,
    "Target excluded": "trend_direction" not in X.columns,
    "trend_pct excluded": "trend_pct" not in X.columns,
    "client_id excluded": "client_id" not in X.columns,
    "Preprocessor created": preprocessor is not None,
    "Feature vector created": X_vector is not None,
}

all_passed = True

for name, result in checks.items():
    print(("PASS" if result else "FAIL") + " — " + name)
    if not result:
        all_passed = False

print(
    "\n"
    + ("ALL CHECKS PASSED" if all_passed else "SOME CHECKS FAILED")
)

ML-05 FINAL SELF-CHECK

PASS — Dataset has 30,000 rows
PASS — Dataset has 44 columns
PASS — Target exists
PASS — Raw feature count is 41
PASS — Target excluded
PASS — trend_pct excluded
PASS — client_id excluded
PASS — Preprocessor created
PASS — Feature vector created

ALL CHECKS PASSED
